# Tydzień 2: Regresja – eksploracyjna analiza danych (EDA) 🔍

| | |
|---|---|
| **Rozdział książki** | 2. *Regression* – podrozdziały: *The Linear Regression Model*, *Loading the Dataset*, *Exploratory Data Analysis* |
| **Czas zajęć** | 90 minut |
| **Dane** | `insurance` – koszty ubezpieczenia zdrowotnego (wbudowane w PyCaret) |
| **Biblioteki** | pandas, Matplotlib, Seaborn |

## 🎯 Cel zajęć
Po dzisiejszych zajęciach będziesz:
1. wiedzieć, czym jest **regresja** i **regresja liniowa** (*linear regression*),
2. rozumieć, po co robimy **eksploracyjną analizę danych** (*Exploratory Data Analysis, EDA*),
3. umieć narysować **histogram**, **wykres słupkowy**, **wykres pudełkowy**, **wykres rozrzutu** i **mapę korelacji**,
4. umieć wyciągnąć z wykresów proste wnioski: *co najbardziej wpływa na koszt ubezpieczenia?*

## 🗓️ Plan zajęć
| Czas | Co robimy |
|---|---|
| 0–15 min | 📚 Teoria: regresja i EDA |
| 15–65 min | 👣 Krok po kroku: wykresy i wnioski |
| 65–85 min | 🧑‍💻 Ćwiczenie samodzielne – ✅ **obowiązkowe** (zaliczenie) |
| 85–90 min | 📝 Podsumowanie |
| poza zajęciami | 🏋️ Część 4: ćwiczenia dodatkowe (trudniejsze, nieobowiązkowe) – dla szybszych osób lub do domu |

---
# 📚 Część 1: Wprowadzenie teoretyczne

## Co to jest regresja?
**Regresja** (*regression*) to zadanie uczenia nadzorowanego, w którym przewidujemy **liczbę**
(wartość ciągłą, *continuous value*). Przykłady: cena mieszkania, czas dojazdu, koszt ubezpieczenia.

## Regresja liniowa – „linia najlepszego dopasowania”
Najprostszy model regresji to **regresja liniowa** (*linear regression*). Wyobraź sobie wykres, na którym każda kropka
to jedno mieszkanie: na osi poziomej metraż, na pionowej cena. Regresja liniowa rysuje **prostą, która przechodzi jak najbliżej
wszystkich kropek**. Dzięki tej prostej możemy odczytać przewidywaną cenę dla dowolnego metrażu.

Wzór (nie przejmuj się – to tylko „przepis” na prostą ze szkoły: *y = a·x + b*):

$$\text{koszt} = \beta_0 + \beta_1 \cdot \text{wiek} + \beta_2 \cdot \text{BMI} + \dots + \varepsilon$$

- $\beta_0$ – wyraz wolny (*intercept*): wartość startowa,
- $\beta_1, \beta_2, \dots$ – współczynniki (*coefficients*): „o ile rośnie koszt, gdy cecha rośnie o 1”,
- $\varepsilon$ – błąd (*error*): to, czego model nie umie wyjaśnić.

Model **sam uczy się** wartości $\beta$ z danych. Oprócz regresji liniowej istnieje wiele innych modeli (drzewa decyzyjne,
lasy losowe, *gradient boosting*…) – PyCaret porówna je za nas w przyszłym tygodniu.

## Po co EDA?
🩺 Lekarz najpierw rozmawia z pacjentem i robi badania, a dopiero potem stawia diagnozę.
Podobnie my: zanim zbudujemy model, musimy **poznać dane**. EDA odpowiada na pytania:
- Czy są braki danych? Jakie są typy kolumn?
- Jak rozkładają się wartości? Czy są wartości nietypowe?
- Które cechy są powiązane ze zmienną celu?

## Typy zmiennych
- **Liczbowe** (*numeric*): wiek, BMI, liczba dzieci, koszt → rysujemy **histogramy**, **wykresy rozrzutu**.
- **Kategoryczne** (*categorical*): płeć, palenie, region → rysujemy **wykresy słupkowe**.

## Korelacja (*correlation*)
Liczba od **−1 do 1**, która mówi, jak mocno dwie zmienne „idą w parze”:
- **bliska 1** – gdy jedna rośnie, druga też rośnie (np. wzrost i rozmiar buta),
- **bliska 0** – brak liniowego związku,
- **bliska −1** – gdy jedna rośnie, druga maleje (np. prędkość jazdy i czas dojazdu).

---
# 👣 Część 2: Krok po kroku

### Krok 1: Importujemy biblioteki

In [ ]:
# Wyłączamy ostrzeżenia (warnings), żeby nie zaśmiecały ekranu.
# Ostrzeżenie to NIE błąd – to tylko podpowiedź dla programistów bibliotek.
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# pandas – praca z tabelami (DataFrame)
import pandas as pd

In [ ]:
# matplotlib – podstawowa biblioteka do wykresów; seaborn – ładniejsze wykresy statystyczne (zbudowana na matplotlib)
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Funkcja get_data z PyCaret – wczytuje gotowe zbiory danych
from pycaret.datasets import get_data

### Krok 2: Wczytujemy dane i sprawdzamy podstawowe informacje

In [ ]:
# Wczytujemy zbiór insurance (koszty ubezpieczenia) do zmiennej data
data = get_data('insurance')

In [ ]:
# info() pokazuje: liczbę wierszy, nazwy kolumn, liczbę niepustych wartości (Non-Null) i typ danych (Dtype)
data.info()

📖 **Jak czytać `Dtype`?**
- `int64` – liczby całkowite (np. wiek), `float64` – liczby z przecinkiem (np. BMI),
- `object` – tekst (np. `male`/`female`) → to zwykle zmienne **kategoryczne**.

In [ ]:
# isnull().sum() liczy BRAKI DANYCH (missing values) w każdej kolumnie. Same zera = świetnie, brak braków 🙂
data.isnull().sum()

In [ ]:
# Zapisujemy nazwy kolumn w dwóch listach – przydadzą się przy rysowaniu wykresów
numeric = ['age', 'bmi', 'children', 'charges']      # kolumny liczbowe
categorical = ['smoker', 'sex', 'region']            # kolumny kategoryczne

### Krok 3: Rozkład zmiennych liczbowych – histogramy 📊
**Histogram** dzieli zakres wartości na przedziały (*bins*) i pokazuje, ile obserwacji wpada do każdego przedziału.
To jak ustawienie ludzi w rzędach według wzrostu i policzenie, ilu stoi w każdym rzędzie.

In [ ]:
# hist() rysuje histogram dla każdej kolumny; bins=20 – liczba przedziałów, figsize – rozmiar rysunku (szer., wys.)
data[numeric].hist(bins=20, figsize=(12, 8))
plt.show()

🔎 **Co widzimy?**
- `charges` (koszt) jest **prawostronnie skośny** (*right-skewed*): większość osób płaci mało, a nieliczni – bardzo dużo
  (podobnie jak z zarobkami w społeczeństwie). Taki „długi ogon w prawo” może przeszkadzać niektórym modelom – zajmiemy się tym w tygodniu 3.
- `age` – dużo bardzo młodych osób (18–19 lat), reszta rozłożona równo.
- `bmi` – kształt zbliżony do „dzwonu” (rozkład normalny).

In [ ]:
# skew() liczy skośność (skewness): 0 = rozkład symetryczny, > 0 = ogon w prawo, < 0 = ogon w lewo
data[numeric].skew()

### Krok 4: Zmienne kategoryczne – wykresy słupkowe

In [ ]:
# value_counts() liczy osoby w każdej kategorii, a plot(kind='bar') rysuje z tego wykres słupkowy (bar chart)
data['smoker'].value_counts().plot(kind='bar', color=['C0', 'C1'])
plt.show()

In [ ]:
# normalize=True zamienia liczebności na ułamki – ok. 20% osób to palacze
data['smoker'].value_counts(normalize=True)

In [ ]:
# Seaborn robi to samo jedną funkcją: countplot liczy obserwacje w każdej kategorii
sns.countplot(data=data, x='region')
plt.show()

### Krok 5: Jak cecha kategoryczna wpływa na koszt? 🚬
Teraz najciekawsze: szukamy związku między cechami a **zmienną celu** (`charges`).
Parametr `hue` (ang. *odcień*) koloruje wykres według kategorii.

In [ ]:
# Histogram kosztów, pokolorowany według palenia; multiple='stack' układa słupki jeden na drugim
sns.histplot(data=data, x='charges', hue='smoker', multiple='stack')
plt.show()

📦 **Wykres pudełkowy** (*box plot*) – jak czytać:
- gruba linia w środku pudełka = **mediana** (wartość środkowa),
- pudełko = środkowe 50% obserwacji,
- „wąsy” = typowy zakres wartości,
- pojedyncze punkty poza wąsami = **obserwacje odstające** (*outliers*).

In [ ]:
# Wykres pudełkowy kosztów osobno dla palących (yes) i niepalących (no)
sns.boxplot(data=data, x='smoker', y='charges')
plt.show()

In [ ]:
# groupby = "pogrupuj według"; liczymy ŚREDNI koszt w każdej grupie
data.groupby('smoker')['charges'].mean()

🔎 **Wniosek:** palacze płacą średnio **prawie 4 razy więcej** niż osoby niepalące! To prawdopodobnie najważniejsza cecha.

### Krok 6: Związek między zmiennymi liczbowymi – wykres rozrzutu
**Wykres rozrzutu** (*scatter plot*): każda kropka to jedna osoba; położenie kropki wyznaczają dwie liczby (np. wiek i koszt).

In [ ]:
# Wiek (oś X) vs. koszt (oś Y); kolor kropki zależy od palenia
sns.scatterplot(data=data, x='age', y='charges', hue='smoker')
plt.show()

🔎 Widać **trzy „pasma”** kropek: koszt rośnie z wiekiem, ale palacze (niebieskie) są zawsze wyżej.

In [ ]:
# regplot dorysowuje prostą regresji liniowej – "linię najlepszego dopasowania" (line of best fit)
sns.regplot(data=data, x='age', y='charges', scatter_kws={'s': 10}, line_kws={'color': 'red'})
plt.show()

💡 Czerwona linia to właśnie **regresja liniowa z jedną cechą**: koszt ≈ a · wiek + b.
Widać, że jedna prosta słabo pasuje do trzech „pasm” – dlatego potrzebujemy więcej cech i lepszych modeli.

In [ ]:
# pairplot rysuje wykresy rozrzutu dla KAŻDEJ pary kolumn naraz; na przekątnej – rozkład każdej zmiennej
sns.pairplot(data[['age', 'bmi', 'charges', 'smoker']], hue='smoker')
plt.show()

🔎 Spójrz na wykres `bmi` vs `charges`: osoby z wysokim BMI płacą dużo **głównie wtedy, gdy palą**.
Takie „współdziałanie” dwóch cech nazywamy **interakcją** (*interaction*).

### Krok 7: Korelacja i mapa ciepła 🌡️

In [ ]:
# corr() liczy współczynnik korelacji Pearsona dla każdej pary kolumn liczbowych
corr = data[numeric].corr()
corr

In [ ]:
# heatmap rysuje tabelę korelacji kolorami; annot=True wpisuje liczby, vmin/vmax ustalają zakres skali kolorów
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.show()

In [ ]:
# Korelacja działa tylko na liczbach. Zamieniamy 'yes'/'no' na 1/0 i liczymy korelację palenia z kosztem.
smoker_01 = (data['smoker'] == 'yes').astype(int)
smoker_01.corr(data['charges'])

## 🧾 Wnioski z EDA
1. **Palenie** ma najsilniejszy związek z kosztem (korelacja ≈ 0,79).
2. **Wiek** – koszt rośnie z wiekiem (korelacja ≈ 0,30).
3. **BMI** – słabszy związek (≈ 0,20), ale wysoki u palaczy z otyłością.
4. Płeć i region mają niewielki wpływ.
5. Rozkład kosztów jest skośny – warto go „wyprostować” (zrobi to za nas PyCaret).

Te wnioski sprawdzimy za dwa tygodnie, gdy model pokaże nam **ważność cech** (*feature importance*).

---
# 🧑‍💻 Część 3: Ćwiczenie dla Ciebie
> ✅ **Ćwiczenie obowiązkowe** – na jego podstawie zaliczasz dzisiejsze zajęcia (zaliczone / niezaliczone).

**Zadanie:** Zbadaj samodzielnie zbiór `insurance`.

- **a)** Narysuj histogram `bmi` z podziałem kolorem na płeć (`sex`).
- **b)** Narysuj wykres rozrzutu: `bmi` (oś X) vs `charges` (oś Y), kolor według `smoker`.
  Co dzieje się z kosztami palaczy, gdy BMI przekracza 30?
- **c)** Policz średni koszt ubezpieczenia w każdym regionie (`region`). Który region jest najdroższy?

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- a) `sns.histplot(data=data, x='...', hue='...')`
- b) `sns.scatterplot(data=data, x='...', y='...', hue='...')`
- c) `data.groupby('...')['charges'].mean()` – a jeśli chcesz posortować: dodaj `.sort_values()`

</details>

In [ ]:
# ✍️ Twój kod tutaj (możesz dodawać kolejne komórki przyciskiem "+ Code")

### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

---
# 🏋️ Część 4: Ćwiczenia dodatkowe – trochę trudniejsze
Dla osób, które skończyły ćwiczenie podstawowe, albo **do samodzielnej pracy w domu**.
Ćwiczenia z tej części są **nieobowiązkowe** – nie są wymagane do zaliczenia zajęć.

Te zadania mają mniej podpowiedzi w treści, łączą kilka poznanych narzędzi i czasem wprowadzają **jedną nową funkcję**
(zawsze opisaną w podpowiedzi). Poziomy trudności: ⭐⭐ – średnie, ⭐⭐⭐ – wyzwanie.

## ⭐⭐ Ćwiczenie 2: Kategorie BMI i palenie
Lekarze dzielą BMI na kategorie: **niedowaga** (< 18,5), **norma** (18,5–25), **nadwaga** (25–30), **otyłość** (≥ 30).

- **a)** Dodaj do tabeli `data` nową kolumnę `bmi_category` z tymi kategoriami.
- **b)** Policz, ile osób jest w każdej kategorii.
- **c)** Policz **średni koszt** ubezpieczenia dla każdej kombinacji *kategoria BMI × palenie* i pokaż wynik jako tabelę
  (wiersze = kategorie BMI, kolumny = `no` / `yes`) oraz jako wykres słupkowy.
- **d)** W której grupie koszty są najwyższe? Czy otyłość „podnosi” koszt tak samo u palących i niepalących?

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- Nowa funkcja: `pd.cut(kolumna, bins=[0, 18.5, 25, 30, 100], labels=[...], right=False)` dzieli liczby na przedziały
  (`right=False` oznacza, że 25,0 trafi do „nadwagi”, a nie do „normy”).
- Grupowanie po **dwóch** kolumnach: `data.groupby(['bmi_category', 'smoker'])['charges'].mean()`.
- `.unstack()` zamienia drugi poziom grupowania (`smoker`) na kolumny; z takiej tabeli `.plot(kind='bar')` rysuje słupki obok siebie.

</details>

In [ ]:
# ✍️ Twój kod tutaj

#### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

## ⭐⭐⭐ Ćwiczenie 3: Wiek, palenie i siła związku
- **a)** Utwórz kolumnę `age_group` z grupami wiekowymi: 18–29, 30–39, 40–49, 50–59, 60–64.
- **b)** Narysuj **jeden** wykres pudełkowy kosztów dla każdej grupy wiekowej, z osobnymi pudełkami dla palących i niepalących.
- **c)** Policz korelację wieku z kosztem **osobno** dla niepalących i dla palących. W której grupie wiek „mocniej” wpływa na koszt? Dlaczego?

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- a) `pd.cut(data['age'], bins=[17, 29, 39, 49, 59, 64], labels=['18-29', '30-39', '40-49', '50-59', '60-64'])` –
  domyślnie przedziały są prawostronnie domknięte, więc (17, 29] obejmuje wiek 18–29.
- b) `sns.boxplot(data=..., x='age_group', y='charges', hue='smoker')`.
- c) Najpierw filtr, np. `non_smokers = data[data['smoker'] == 'no']`, potem `non_smokers['age'].corr(non_smokers['charges'])`.

</details>

In [ ]:
# ✍️ Twój kod tutaj

#### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

---
# 📝 Podsumowanie
- **Regresja** przewiduje liczbę; **regresja liniowa** dopasowuje do danych prostą.
- **EDA** to „badanie danych” przed budową modelu: typy kolumn, braki, rozkłady, związki ze zmienną celu.
- Narzędzia: `info()`, `isnull().sum()`, `hist()`, `value_counts()`, `sns.histplot`, `sns.boxplot`, `sns.scatterplot`,
  `sns.pairplot`, `corr()` + `sns.heatmap`.
- Najważniejsze cechy w zbiorze `insurance`: **palenie** i **wiek**.

## 📖 Słowniczek PL / EN
| Polski | English | Znaczenie |
|---|---|---|
| eksploracyjna analiza danych | Exploratory Data Analysis (EDA) | poznawanie danych wykresami i statystykami |
| regresja liniowa | linear regression | model dopasowujący prostą do danych |
| współczynnik | coefficient | „waga” cechy w modelu liniowym |
| histogram | histogram | wykres rozkładu wartości w przedziałach |
| skośność | skewness | asymetria rozkładu („długi ogon”) |
| wykres rozrzutu | scatter plot | kropki dla dwóch zmiennych liczbowych |
| wykres pudełkowy | box plot | mediana, kwartyle i wartości odstające |
| korelacja | correlation | siła liniowego związku dwóch zmiennych (−1…1) |
| obserwacja odstająca | outlier | wartość wyraźnie inna niż pozostałe |
| brakujące dane | missing values | puste komórki w tabeli |

➡️ **Za tydzień:** przygotujemy dane funkcją `setup()` i porównamy ok. 20 modeli regresji jednym poleceniem `compare_models()`.